In [1]:
import pandas as pd

df = pd.read_csv("electricity_bill_dataset.csv")

print(df.head())
print(df.shape)

   Fan  Refrigerator  AirConditioner  Television  Monitor  MotorPump  Month  \
0   16          23.0             2.0         6.0      1.0          0     10   
1   19          22.0             2.0         3.0      1.0          0      5   
2    7          20.0             2.0         6.0      7.0          0      7   
3    7          22.0             3.0        21.0      1.0          0      6   
4   11          23.0             2.0        11.0      1.0          0      2   

        City                                    Company  MonthlyHours  \
0  Hyderabad                    Tata Power Company Ltd.           384   
1   Vadodara                                       NHPC           488   
2     Shimla                            Jyoti Structure           416   
3     Mumbai                            Power Grid Corp           475   
4     Mumbai  Ratnagiri Gas and Power Pvt. Ltd. (RGPPL)           457   

   TariffRate  ElectricityBill  
0         8.4           3225.6  
1         7.8       

In [2]:
df = df.drop(["City", "Company"], axis=1)

print(df.head())
print(df.shape)

   Fan  Refrigerator  AirConditioner  Television  Monitor  MotorPump  Month  \
0   16          23.0             2.0         6.0      1.0          0     10   
1   19          22.0             2.0         3.0      1.0          0      5   
2    7          20.0             2.0         6.0      7.0          0      7   
3    7          22.0             3.0        21.0      1.0          0      6   
4   11          23.0             2.0        11.0      1.0          0      2   

   MonthlyHours  TariffRate  ElectricityBill  
0           384         8.4           3225.6  
1           488         7.8           3806.4  
2           416         7.7           3203.2  
3           475         9.2           4370.0  
4           457         9.2           4204.4  
(45345, 10)


In [3]:
X = df.drop("ElectricityBill", axis=1)

y = df["ElectricityBill"]

print(X.head())
print()
print(y.head())

   Fan  Refrigerator  AirConditioner  Television  Monitor  MotorPump  Month  \
0   16          23.0             2.0         6.0      1.0          0     10   
1   19          22.0             2.0         3.0      1.0          0      5   
2    7          20.0             2.0         6.0      7.0          0      7   
3    7          22.0             3.0        21.0      1.0          0      6   
4   11          23.0             2.0        11.0      1.0          0      2   

   MonthlyHours  TariffRate  
0           384         8.4  
1           488         7.8  
2           416         7.7  
3           475         9.2  
4           457         9.2  

0    3225.6
1    3806.4
2    3203.2
3    4370.0
4    4204.4
Name: ElectricityBill, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(36276, 9)
(9069, 9)


In [18]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

print("Random Forest Model training complete")

Random Forest Model training complete


In [6]:
y_pred = model.predict(X_test)

print(y_pred[:5])

[5034.9778     3856.27592958 3234.68374444 5725.51254099 4152.39859756]


In [7]:
from sklearn.metrics import r2_score

score = r2_score(y_test, y_pred)

print("R2 Score:", score)

R2 Score: 0.9956383663641158


In [21]:
import joblib

joblib.dump(model, "electricity_model.pkl")

['electricity_model.pkl']

In [22]:
import joblib

joblib.dump(X.columns.tolist(), "features.pkl")

print("Features Saved")

Features Saved


In [11]:
import pandas as pd

sample = pd.DataFrame([X_test.iloc[0]], columns=X.columns)

pred = model.predict(sample)

print("Predicted Bill:", pred[0])

Predicted Bill: 5034.977800004346


In [19]:
prediction = model.predict(sample)[0]

if prediction < 0:
    prediction = 0

In [20]:
from sklearn.metrics import r2_score

print("Accuracy:", r2_score(y_test, y_pred))

Accuracy: 0.9956383663641158


In [23]:
import joblib

joblib.dump(model, "electricity_model.pkl")

print("Model saved successfully ✅")

Model saved successfully ✅


In [24]:
joblib.dump(X.columns.tolist(), "features.pkl")

print("Features saved successfully ✅")

Features saved successfully ✅


In [33]:
import tkinter as tk
from tkinter import ttk
import pandas as pd
import joblib

# Load model + features
model = joblib.load("electricity_model.pkl")
features = joblib.load("features.pkl")

# Window setup
root = tk.Tk()
root.title("⚡ Electricity Bill Predictor")
root.geometry("700x600")
root.configure(bg="#0f172a")

# Title
title = tk.Label(
    root,
    text="SMART ELECTRICITY BILL PREDICTOR",
    font=("Helvetica", 20, "bold"),
    bg="#0f172a",
    fg="white"
)
title.pack(pady=20)

# Card frame
card = tk.Frame(root, bg="#1e293b", padx=20, pady=20)
card.pack(pady=10)

entries = {}

# Input fields
for i, feature in enumerate(features):
    label = tk.Label(
        card,
        text=feature,
        font=("Arial", 11),
        bg="#1e293b",
        fg="white"
    )
    label.grid(row=i, column=0, sticky="w", pady=5, padx=10)

    entry = ttk.Entry(card, width=25)
    entry.grid(row=i, column=1, pady=5)

    entries[feature] = entry


# Prediction function
def predict_bill():
    try:
        data = []

        for feature in features:
            value = float(entries[feature].get())
            data.append(value)

        # If all zeros → force 0 bill
        if all(v == 0 for v in data):
            result_label.config(
                text="💰 Predicted Electricity Bill: 0.00",
                fg="#22c55e"
            )
            return

        sample = pd.DataFrame([data], columns=features)

        prediction = model.predict(sample)[0]

        # safety fix (no negative bill)
        if prediction < 0:
            prediction = 0

        result_label.config(
            text=f"💰 Predicted Electricity Bill: {prediction:.2f}",
            fg="#22c55e"
        )

    except:
        result_label.config(
            text="⚠ Please enter valid numbers!",
            fg="red"
        )


# Button
btn = tk.Button(
    root,
    text="PREDICT BILL 💰",
    command=predict_bill,
    font=("Arial", 14, "bold"),
    bg="#22c55e",
    fg="black",
    padx=20,
    pady=8
)
btn.pack(pady=20)

# Result label
result_label = tk.Label(
    root,
    text="",
    font=("Arial", 16, "bold"),
    bg="#0f172a",
    fg="white"
)
result_label.pack(pady=10)

root.mainloop()

In [36]:
root = tk.Tk()
root.title("⚡ Electricity Bill Predictor")

# 👇 YAHAN PASTE KARO (window size fix)
root.update_idletasks()
width = 600
height = 500
x = (root.winfo_screenwidth() // 2) - (width // 2)
y = (root.winfo_screenheight() // 2) - (height // 2)
root.geometry(f"{width}x{height}+{x}+{y}")
root.resizable(False, False)

root.configure(bg="#0f172a")